# SFT → eval → GRPO → final eval

**Requires GPU runtime (T4).** Run all cells top to bottom.

In [ ]:
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project 2>/dev/null || (cd /content/ml_project && git pull)
%cd /content/ml_project
!pip install -q -r requirements-colab.txt
!pip uninstall -y -q bitsandbytes torchvision

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
import transformers, trl, peft
print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)

## 0. Smoke Full Pipeline

Run this first after setup. It does a small SFT, short GRPO, and eval to catch issues before the full run.


In [ ]:
!rm -rf ckpts/smoke_sft ckpts/smoke_grpo
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True   python notebooks/smoke_full_pipeline.py --clean --sft-epochs 1 --grpo-steps 12 --n-val 8 --samples-per-layout 1


## 1. SFT

In [ ]:
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!rm -rf ckpts/sft
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python -m training.train_sft --output-dir ./ckpts/sft --num-train-epochs 1

## 2. Eval policy_0 (base) vs policy_1 (SFT)

In [ ]:
import os, json, pandas as pd
os.makedirs('results', exist_ok=True)
from training.evaluate import evaluate_checkpoints, save_results, rows_to_table
results = evaluate_checkpoints(
    [{'name': 'policy_0', 'adapter': None},
     {'name': 'policy_1', 'adapter': './ckpts/sft'}],
    samples_per_layout=4,
)
print(rows_to_table(results))
save_results(results, 'results/eval_sft_vs_base.json')
d = json.load(open('results/eval_sft_vs_base.json'))
rows = [{'ckpt': c['name'], 'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1),
         'materials': round(c['mean_materials'], 1),
         'machines': round(c['mean_machines'], 2)} for c in d]
pd.DataFrame(rows).set_index('ckpt')

## 3. GRPO

In [ ]:
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python -m training.train_grpo \
  --init-adapter ./ckpts/sft \
  --curriculum \
  --output-dir ./ckpts/grpo \
  --group-size 6 \
  --temperature 1.0 \
  --per-device-batch-size 2 \
  --gradient-accumulation-steps 3 \
  --max-steps 100 \
  --save-steps 25 \
  --max-prompt-length 3500 \
  --max-completion-length 512

## 4. Final eval: policy_0 vs policy_1 vs GRPO checkpoints

In [ ]:
import os, json, pandas as pd
os.makedirs('results', exist_ok=True)
from training.evaluate import evaluate_checkpoints, save_results, rows_to_table
results = evaluate_checkpoints(
    [{'name': 'policy_0', 'adapter': None},
     {'name': 'policy_1', 'adapter': './ckpts/sft'},
     {'name': 'policy_2', 'adapter': './ckpts/grpo/checkpoint-25'},
     {'name': 'policy_3', 'adapter': './ckpts/grpo/checkpoint-50'},
     {'name': 'policy_4', 'adapter': './ckpts/grpo/checkpoint-75'},
     {'name': 'policy_final', 'adapter': './ckpts/grpo'}],
    samples_per_layout=4,
    n_val=40,
)
print(rows_to_table(results))
save_results(results, 'results/eval_final.json')
d = json.load(open('results/eval_final.json'))
rows = [{'ckpt': c['name'], 'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1),
         'materials': round(c['mean_materials'], 1),
         'machines': round(c['mean_machines'], 2)} for c in d]
pd.DataFrame(rows).set_index('ckpt')
